In [ ]:
import cv2
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from moviepy.editor import *

#### Make dataframe

In [ ]:
dirs = glob.glob('./trials/true/*')

In [ ]:
def get_stim(stim_type):
    stim = []
    for dir in dirs:
        if len(glob.glob(f'{dir}/*_{stim_type}_*.mov')) != 0:
            stim.append(glob.glob(f'{dir}/*_{stim_type}_*.mov'))

    return stim


def get_info(filename):
    info = []

    info.append(int(filename.split(os.sep)[1]))
    info.append(int(filename.split(os.sep)[2].split('_')[1]))
    info.append(filename.split(os.sep)[2].split('_')[-1][:1])

    video = cv2.VideoCapture(filename)
    fps = video.get(cv2.CAP_PROP_FPS)
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    info.append(np.round(frame_count/fps))

    info.append(filename[0:-9]+'M.mov')
    info.append(filename[0:-9]+'F.mov')
    info.append('no')

    return info


df = pd.DataFrame(columns=['dyad', 'extract', 'speaker', 'duration', 'm', 'f', 'used'])

stim_list = get_stim('all')
flat_list = [item for sublist in stim_list for item in sublist]
for i in range(len(flat_list)):
    df.loc[i] = get_info(flat_list[i])

In [ ]:
df

In [ ]:
df_dyad = df[df['dyad'] == 9]
df_dyad

#### Remove dyads with fewer than n trials

In [ ]:
n = 4       # minimum number of trials

df['num_trials'] = df.groupby('dyad')['extract'].transform('count')
df_viable = df[df['num_trials'] >= n]

In [ ]:
df_viable

#### Make trials

In [ ]:
def color_clip(size, duration, fps, color=(0,0,0), output='./blank.mov'):
    ColorClip(size, color, duration=duration).write_videofile(output, fps=fps, codec='libx264', audio_codec='aac')

    return output


def stitch_video(clip1, clip2, condition, output):
    if condition == 'vv':
        final_clip = clips_array([[clip1.without_audio(), clip2.without_audio()]])
    elif condition == 'va':
        audio = clip1.audio
        clip2.audio = audio
        dur = clip1.duration
        size = clip1.size
        fps = clip1.fps
        blank_clip = VideoFileClip(color_clip(size, dur, fps))
        final_clip = clips_array([[blank_clip, clip2]])
        final_clip.audio.set_fps(30)
    elif condition == 'vva':
        final_clip = clips_array([[clip1, clip2.without_audio()]])
        final_clip.audio.set_fps(30)
        

    final_clip.write_videofile(output, codec='libx264', audio_codec='aac')


def make_true_trial(df, trial_id, dyad_id):
    speaker = df[df['extract'] == trial_id]['speaker'].values[0]

    if speaker == 'M':
        x = df[df['extract'] == trial_id]['m'].values[0]
        y = df[df['extract'] == trial_id]['f'].values[0]
    elif speaker == 'F':
        x = df[df['extract'] == trial_id]['f'].values[0]
        y = df[df['extract'] == trial_id]['m'].values[0]

    clip1 = VideoFileClip(x)
    clip1.audio.write_audiofile(f"./tests/true/{dyad_id}_{trial_id}_audio.wav", codec='pcm_s32le')
    clip2 = VideoFileClip(y)

    stitch_video(clip1, clip2, 'vv', f"./tests/true/{dyad_id}_{trial_id}_vv.mov")
    stitch_video(clip1, clip2, 'va', f"./tests/true/{dyad_id}_{trial_id}_va.mov")
    stitch_video(clip1, clip2, 'vva', f"./tests/true/{dyad_id}_{trial_id}_vva.mov")


def make_fake_trial(df, trials, dyad_id):
    speaker = df[df['extract'] == trials[0]]['speaker'].values[0]

    if speaker == 'M':
        x = df[df['extract'] == trials[0]]['m'].values[0]
        y = df[df['extract'] == trials[1]]['f'].values[0]
    elif speaker == 'F':
        x = df[df['extract'] == trials[0]]['f'].values[0]
        y = df[df['extract'] == trials[1]]['m'].values[0]

    clip1 = VideoFileClip(x)
    clip1.audio.write_audiofile(f"./tests/fake/{dyad_id}_{trials[0]}_{trials[1]}_audio.wav", codec='pcm_s32le')
    clip2 = VideoFileClip(y)

    stitch_video(clip1, clip2, 'vv', f"./tests/fake/{dyad_id}_{trials[0]}_{trials[1]}_vv.mov")
    stitch_video(clip1, clip2, 'va', f"./tests/fake/{dyad_id}_{trials[0]}_{trials[1]}_va.mov")
    stitch_video(clip1, clip2, 'vva', f"./tests/fake/{dyad_id}_{trials[0]}_{trials[1]}_vva.mov")

In [ ]:
dyad_ids = np.asarray(df_viable['dyad'].unique())
dyad_ids_sorted = np.sort(dyad_ids)

for id in dyad_ids_sorted[:2]:
    df_dyad = df_viable[df_viable['dyad'] == id]
    trials = df_dyad['extract'].values

    if len(df_dyad[df_dyad['speaker'] == 'M'].index) < 2:
        fake_trials = df_dyad[df_dyad['speaker'] == 'F']['extract'].sample(n=2, random_state=44, replace=False).values
    else:
        fake_trials = df_dyad[df_dyad['speaker'] == 'M']['extract'].sample(n=2, random_state=44, replace=False).values

    true_trial = np.random.choice(list(set(trials) - set(fake_trials)), size=1)[0]

    make_true_trial(df_dyad, true_trial, id)
    make_fake_trial(df_dyad, fake_trials, id)